# Score and repair prompts with Claude

Every new Claude user writes weak prompts at first — but the hard part isn't fixing them, it's **knowing what's wrong**. This recipe builds a small, reusable toolkit that:

1. **Scores** a draft prompt against five prompt-engineering best practices and explains *why* each one is weak,
2. **Repairs** it — rewriting the rough draft into a well-structured prompt, and
3. **Proves it worked** — re-scoring the rewrite and measuring the lift across a batch.

The result is two drop-in helpers, `score()` and `improve()`, you can paste into your own app to grade and upgrade your users' prompts before they ever hit the model.

Input:  `fix my resume idk make it good`
→ diagnosis + a rewritten prompt that measurably scores higher.

## Setup

You'll need an Anthropic API key set as the `ANTHROPIC_API_KEY` environment variable.

In [ ]:
%pip install -q anthropic

from anthropic import Anthropic

client = Anthropic()  # reads ANTHROPIC_API_KEY from the environment

# Use the latest Sonnet available in your account.
MODEL = "claude-sonnet-4-5"

## Part 1 — Score a prompt

We grade a draft on five dimensions drawn from Anthropic's prompt-engineering guidance, each on a 0–2 scale:

| Dimension | Question |
|---|---|
| **role** | Is Claude given a persona or expertise to answer from? |
| **context** | Is there background or situational grounding? |
| **specificity** | Is the ask concrete (counts, constraints, criteria, examples)? |
| **format** | Is the desired output shape stated (bullets, table, JSON, length)? |
| **structure** | For longer prompts, is it organized (sections, delimiters, tags)? |

To get a reliable, machine-readable result we use **tool use** to force Claude's answer into a fixed schema — no brittle string parsing.

In [ ]:
SCORE_TOOL = {
    "name": "record_prompt_score",
    "description": "Record a structured quality score for a user's draft prompt.",
    "input_schema": {
        "type": "object",
        "properties": {
            "dimensions": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "name": {
                            "type": "string",
                            "enum": ["role", "context", "specificity", "format", "structure"],
                        },
                        "score": {
                            "type": "integer",
                            "description": "0 = absent, 1 = partial, 2 = strong",
                        },
                        "reason": {"type": "string", "description": "one concise sentence"},
                    },
                    "required": ["name", "score", "reason"],
                },
            }
        },
        "required": ["dimensions"],
    },
}

SCORE_SYSTEM = (
    "You are a strict but fair prompt-engineering evaluator. Score the user's "
    "DRAFT prompt on five dimensions (role, context, specificity, format, "
    "structure), each 0-2: 0 absent, 1 partial, 2 strong. Judge the draft as "
    "written, not the topic. Call record_prompt_score exactly once."
)


def score(draft: str) -> dict:
    """Return {'total': 0-100, 'dimensions': {name: {'score', 'reason'}}}."""
    msg = client.messages.create(
        model=MODEL,
        max_tokens=1024,
        system=SCORE_SYSTEM,
        tools=[SCORE_TOOL],
        tool_choice={"type": "tool", "name": "record_prompt_score"},
        messages=[{"role": "user", "content": f"<draft>{draft}</draft>"}],
    )
    dims = {}
    for block in msg.content:
        if block.type == "tool_use":
            for d in block.input["dimensions"]:
                dims[d["name"]] = {"score": d["score"], "reason": d["reason"]}
    total = round(100 * sum(d["score"] for d in dims.values()) / (2 * len(dims))) if dims else 0
    return {"total": total, "dimensions": dims}

In [ ]:
def show_score(label: str, result: dict) -> None:
    print(f"{label}: {result['total']}/100")
    for name, d in result["dimensions"].items():
        mark = {2: "✓", 1: "~", 0: "✗"}[d["score"]]
        print(f"  {mark} {name:<12} {d['score']}/2  {d['reason']}")


weak = "fix my resume idk make it good"
show_score("Draft", score(weak))

You'll see a low total with a one-line reason for each weak dimension — the diagnosis a beginner is missing.

## Part 2 — Repair the prompt

Now a single call rewrites the rough draft into a structured prompt, preserving the user's intent and topic while adding the ingredients the score flagged as missing.

In [ ]:
IMPROVE_SYSTEM = (
    "You are an expert prompt engineer. Rewrite the user's rough DRAFT into a "
    "single, well-structured prompt they could send to Claude to get an "
    "excellent result. Preserve their intent and topic. Add a clear role, "
    "any needed context as bracketed placeholders, a specific ask, and an "
    "explicit output format. Return ONLY the rewritten prompt text - no "
    "preamble, no explanation, no quotes."
)


def improve(draft: str) -> str:
    """Rewrite a rough draft into a structured prompt."""
    msg = client.messages.create(
        model=MODEL,
        max_tokens=1024,
        system=IMPROVE_SYSTEM,
        messages=[{"role": "user", "content": f"<draft>{draft}</draft>"}],
    )
    return "".join(b.text for b in msg.content if b.type == "text").strip()


improved = improve(weak)
print(improved)

## Part 3 — Prove it worked

Diagnose-and-repair is only worth anything if the repair actually helps. Re-score the rewrite and compare.

In [ ]:
before = score(weak)
after = score(improved)

show_score("Before", before)
print()
show_score("After ", after)
print(f"\nLift: {after['total'] - before['total']:+d} points")

## Part 4 — Does it hold up across many prompts?

One example is an anecdote. Run the full **score → improve → re-score** loop over a batch of realistically-bad prompts and report the average improvement — a small eval, not a vibe check.

In [ ]:
examples = [
    "fix my resume idk make it good",
    "write something about climate change",
    "help me with my code its broken",
    "summarize this for me",
    "give me some marketing ideas",
]

rows = []
for draft in examples:
    b = score(draft)["total"]
    a = score(improve(draft))["total"]
    rows.append((draft, b, a, a - b))

print(f"{'draft':<40}{'before':>8}{'after':>8}{'lift':>7}")
print("-" * 63)
for draft, b, a, lift in rows:
    print(f"{draft[:38]:<40}{b:>8}{a:>8}{lift:>+7}")

avg = sum(r[3] for r in rows) / len(rows)
print("-" * 63)
print(f"Average improvement: {avg:+.1f} points across {len(rows)} prompts")

## Part 5 — The reusable toolkit

That's the whole recipe. `score()` and `improve()` are self-contained — drop them into any app to grade a user's prompt, tell them what's missing, and offer a one-click upgrade:

```python
result = score(user_prompt)
if result["total"] < 60:
    better = improve(user_prompt)      # offer this as a suggestion
```

### Where to take it next
- **Cache the system prompts** with prompt caching to cut latency and cost on the hot path.
- **Stream** `improve()` so the rewrite appears token-by-token in your UI.
- **Tune the rubric** to your product — add a "safety" or "brand voice" dimension by extending the tool schema.

Prompt quality is the cheapest lever you have on output quality: a better prompt means fewer wasted round-trips and a better answer the first time.